In [93]:
from selenium import webdriver
# from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
# from selenium.webdriver.chrome.options import Options
import chromedriver_autoinstaller
from pydantic import BaseModel, AnyUrl, ValidationError
import time


chromedriver_autoinstaller.install()  # Instala o ChromeDriver automaticamente


'c:\\Users\\user\\AppData\\Local\\Programs\\Python\\Python312\\Lib\\site-packages\\chromedriver_autoinstaller\\139\\chromedriver.exe'

In [94]:
class Imovel(BaseModel):
    title: str
    street: str
    link: AnyUrl | None
    price: float
    location: str
    rooms: int
    area: float
    bathrooms: int
    

def get_important_data(element):
    # Link do imóvel
    link = element.find_element(By.TAG_NAME, "a").get_attribute("href")
    
    # Título do imóvel (ex: "Sala/Conjunto para alugar com 95 m², 1 banheiro, 1 vaga em Jardim, Santo André")
    title = element.find_element(By.TAG_NAME, "a").get_attribute("title")
    
    # Localização (bairro/cidade)
    location = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-location-txt"]').text.strip()
    location = location.split("\n")[1] if "\n" in location else location
    
    # Endereço (rua)
    try:
        street = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-street-txt"]').text.strip()
    except:
        street = ""
    
    # Área
    try:
        area_txt = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-propertyArea-txt"]').text
        area = float(area_txt.split("\n")[-1].replace("m²", "").replace(",", ".").strip())
    except:
        area = 0.0

    # Banheiros
    try:
        bathrooms_txt = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-bathroomQuantity-txt"]').text
        bathrooms = int(bathrooms_txt.split("\n")[-1])
    except:
        bathrooms = 0

    # Vagas
    try:
        rooms_txt = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-parkingSpacesQuantity-txt"]').text
        rooms = int(rooms_txt.split("\n")[-1])
    except:
        rooms = 0

    # Preço
    try:
        price_txt = element.find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-price-txt"] p').text
        price = float(price_txt.split("\n")[-1].replace("R$", "").replace(".", "").replace(",", ".").split("/")[0])
    except:
        price = 0.0
    
    try:
        object_imovel = Imovel(
            title=title,
            street=street,
            link=link,
            price=price,
            location=location,
            rooms=rooms,
            area=area,
            bathrooms=bathrooms
        )
    except ValidationError as e:
        print(e.errors())
        object_imovel = None
    return object_imovel

In [95]:
# Configurações do Chrome
# chrome_options = Options()
# chrome_options.add_argument("--headless")  # Executa sem abrir janela
# chrome_options.add_argument("--disable-gpu")
# chrome_options.add_argument("--window-size=1920,1080")
# chrome_options.add_argument("--no-sandbox")
# chrome_options.add_argument("--disable-dev-shm-usage")

# # Caminho para o chromedriver (ajuste se necessário)
# service = Service('chromedriver.exe')

# Inicializa o navegador
driver = webdriver.Chrome()

# URL alvo
url = "https://www.zapimoveis.com.br/aluguel/"
driver.get(url)


In [96]:

# Usando XPath
# lista_imoveis = driver.find_element(By.XPATH, "/html/body/section/div/div[3]/div[4]")

# Ou usando CSS Selector
# O motivo de não conseguir acessar diretamente "ul" com a classe "flex flex-col gap-3" usando find_element(By.CLASS_NAME, ...) é porque o método By.CLASS_NAME espera apenas um nome de classe, e não múltiplos nomes separados por espaço. 
# Para buscar por múltiplas classes, utilize o seletor CSS:

lista_imoveis = driver.find_elements(By.CSS_SELECTOR, '[data-cy="rp-property-cd"]')

In [97]:
lista_imoveis[0].find_element(By.CSS_SELECTOR, '[data-cy="rp-cardProperty-price-txt"] p').text

'R$ 5.500/mês'

In [98]:
lista = [get_important_data(links) for links in lista_imoveis]

In [99]:
with open("imoveis.txt", "w", encoding="utf-8") as file:
    for object_imovel in lista:
        file.write(object_imovel.model_dump_json(indent=4))
        file.write("\n")

In [100]:
driver.quit()  # Fecha o navegador